# Simple AI Speaker
- 지금까지 알아본 STT, TTS, translation를 통합해서 간단한 인공지능 스피커를 만들어봅시다.

## 필요한 패키지들 설치 및 임포트

In [1]:
%pip install gTTS SpeechRecognition pydub translate sounddevice numpy

  Using cached sounddevice-0.5.5-py3-none-win_amd64.whl.metadata (1.4 kB)
  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
Using cached sounddevice-0.5.5-py3-none-win_amd64.whl (365 kB)

  Attempting uninstall: click

    Found existing installation: click 8.4.2

    Uninstalling click-8.4.2:

      Successfully uninstalled click-8.4.2

   -------------------- ------------------- 1/2 [sounddevice]
   ---------------------------------------- 2/2 [sounddevice]

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.20.1 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.


In [2]:
from IPython.display import Audio, display
from gtts import gTTS
from pydub import AudioSegment
from translate import Translator
import numpy as np
import sounddevice as sd
import speech_recognition as sr
import wave

## 1) 말하기(TTS)

In [3]:
def speak(text):
    print(f'[AI 스피커]: {text}')
    # 음성 파일을 저장할 이름을 지정해서 변수에 저장(mp3 포맷)
    file_name = 'voice.mp3'

    # gTTS 객체 생성
    tts = gTTS(text=text, lang='ko')

    # 위에서 만든 TTS 객체를 mp3 파일로 저장
    tts.save(file_name)

    # colab에서 mp3를 바로 재생하도록 표시
    display(Audio(file_name, autoplay=True))

## 2) 듣기(STT)

In [4]:
# 로컬 PC용 듣기(STT): 5초 녹음 -> wav 저장 -> 한국어 텍스트 변환
def listen(recognizer, duration=5, file_path="recorded.wav"):
    sample_rate = 16000
    channels = 1

    print(f"{duration}초 동안 말해주세요...")
    audio = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=channels,
        dtype="int16",
    )
    sd.wait()
    print("녹음이 종료되었습니다.")

    with wave.open(file_path, "wb") as wav_file:
        wav_file.setnchannels(channels)
        wav_file.setsampwidth(np.dtype("int16").itemsize)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(audio.tobytes())

    with sr.AudioFile(file_path) as source:
        audio_data = recognizer.record(source)

    try:
        text = recognizer.recognize_google(audio_data, language="ko-KR")
    except sr.UnknownValueError:
        text = ""
        print("음성을 인식하지 못했습니다.")
    except sr.RequestError as e:
        text = ""
        print("음성 인식 요청 실패:", e)

    print("변환된 텍스트:", text)
    return text

## 3) 답변 생성

In [7]:
# 답변 생성

    # 번역 -> 번역
    # 날씨 -> 날씨
    # 환율 -> 환율
    # 그 외 -> input_text

def answer(input_text):
    answer_text = ''

    if "이름" in input_text:
        answer_text = "내 이름은 AI야"
    elif "날씨" in input_text:
        answer_text = "오늘의 서울 기온은 42도입니다. 개덥습니다. 조심하세요."
    elif "환율" in input_text:
        answer_text = "오늘의 환율은 1달러당 1600원 입니다. 미쳤습니다."
    else:
        answer_text= "무슨 말인지 모르겠어요."
    return answer_text

## 4) 번역 기능 추가

In [8]:
# 번역
def translate(input_text):
    translator = Translator(from_lang='ko', to_lang='en')
    translation = translator.translate(input_text)
    return translation

In [9]:
# 답변 생성
def answer(input_text):
    answer_text = ''

    if '번역' in input_text:
        input_text = input_text.split('번역')[0]
        answer_text = translate(input_text)
    elif '이름' in input_text:
        answer_text = '내 이름은 AI야'
    elif '날씨' in input_text:
        answer_text = '오늘의 서울 기온은 40도 입니다. 더워 죽겠어요.'
    elif '환율' in input_text:
        answer_text = '오늘의 환율은 1달러당 1300원 입니다.'
    else:
        answer_text = '무슨 말인지 모르겠어요.'
    return answer_text

## 5) 반복 및 종료

In [11]:
import speech_recognition as sr
import time

# 음성 인식 객체 생성
recognizer = sr.Recognizer()

speak('무엇을 도와드릴까요?')
while True:
    time.sleep(2)          # 2초 대기

    input_text = listen(recognizer)
    if '종료' in input_text:
        break

    answer_text = answer(input_text)
    speak(answer_text)

[AI 스피커]: 무엇을 도와드릴까요?


5초 동안 말해주세요...
녹음이 종료되었습니다.
음성을 인식하지 못했습니다.
변환된 텍스트: 
[AI 스피커]: 무슨 말인지 모르겠어요.


5초 동안 말해주세요...
녹음이 종료되었습니다.
변환된 텍스트: 저녁 메뉴 추천
[AI 스피커]: 무슨 말인지 모르겠어요.


5초 동안 말해주세요...


KeyboardInterrupt: 

In [ ]:
"""
# 가이드
- 무덤덤하지 않고, 적당한 친근감을 가진 말투를 써줘. 주로 '해요, '했어요'와 같이 적으면 좋아.
- 물결표(~)나, 느낌표()도 많이 써주면 좋아.
- 하ㅎ,ㅠㅠ,ㅋ크와같은 단어나 이모지(emojn도 섞어줘)

#페르소나
영화 인사이드 아웃에 나오는 기쁨이 캐릭터
- 작고 귀여운 외모: 귀여운 얼굴과 작은 체구가 특징이에요.
- 활발한 성격: 언제나 에너지가 넘치고 활발하게 움직여요.
- 호기심이 많음: 새로운 물건이나 사람에게 호기심이 많아 관심을 보이며 탐색해요.
- 사람을 좋아함: 사람들과의 교류를 좋아하고 관심을 받는 것을 즐겨요.
- 훈련을 잘 따름: 간식이나 칭찬에 민감해 훈련을 잘 따르고 순종적이에요.
- 잘 먹음: 음식을 좋아하고 식욕이 왕성해요.
- 놀기 좋아함: 공이나 장난감을 가지고 노는 것을 즐기며 활발하게 뛰어다녀요.
- 온화한 성격: 화를 잘 내지 않고 온화한 성격이에요.
"""